# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule in plain words:** a page goes to the refresh queue if it hasn't been updated in
**180+ days** and still earns **real search visibility** (≥100 impressions in the trailing
90 days); the queue is ranked by impressions, so editors see the biggest stale pages first.
One reason code: **`STALE_VISIBLE`**. One action label: **`REFRESH`**.

Before coding it, the two signals it leans on get checked — with verdicts from the tables,
not from hope. Signal 1 (staleness) sits behind FlyRank's real refresh flags from the
session; signal 2 is the CTR-vs-position idea behind the CTR-fix logic.

**Signal 1 — staleness → decline: verdict MIXED.** Decline rises from 0.512 (fresh ≤90d,
n=20,655) to 0.611 in the 91–180d band (n=9,171) — but then *drops* to 0.467 in 181–365d on
a starving n=169. Staleness is not monotone on its own; it earns its place only paired with
visibility (checked in the rule cell below). A clearly-explained MIXED just saved the rule
from a naive "the staler the worse" story.

**Signal 2 — CTR below its position band → decline: verdict CONFIRMED (directional).**
On rows with real position data (`avg_position > 0`; zeros mean "no data"), pages earning
less than half their position band's median CTR decline at **0.613 vs 0.547** for the rest
(+6.6 pp over n=7,636). Real, modest, and kept out of today's single rule — it becomes a
model feature next week.

In [1]:
# Setup + Signal 1: decline rate by staleness bucket (bucket table, n printed).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"].str.lower() == "down").astype(int)   # evaluation label ONLY
print(f"{len(df):,} pages | base decline rate {y.mean():.3f}\n")

stale_b = pd.cut(df["days_since_last_update"], [-1, 90, 180, 365, 10_000],
                 labels=["<=90d", "91-180d", "181-365d", ">365d"])
sig1 = pd.DataFrame({"decline_rate": y.groupby(stale_b, observed=True).mean().round(3),
                     "n": y.groupby(stale_b, observed=True).size()})
print("Signal 1 — staleness buckets:")
print(sig1.to_string())
print("\nVerdict: MIXED (rises to 91-180d, non-monotone after; tiny n past 180d alone)")

30,000 pages | base decline rate 0.542

Signal 1 — staleness buckets:
                        decline_rate      n
days_since_last_update                     
<=90d                          0.512  20655
91-180d                        0.611   9171
181-365d                       0.467    169
>365d                          0.600      5

Verdict: MIXED (rises to 91-180d, non-monotone after; tiny n past 180d alone)


In [2]:
# Signal 2: CTR under the position band's median (rows with real position data only).
pos = df.loc[df["avg_position"] > 0].copy()          # avg_position == 0 means NO DATA, not rank 0
ypos = y.loc[pos.index]
pos["band"] = pd.cut(pos["avg_position"], [0, 3, 10, 20, 1000],
                     labels=["top3", "4-10", "11-20", "21+"])
band_median = pos.groupby("band", observed=True)["ctr"].transform("median")
low_ctr = pos["ctr"] < 0.5 * band_median

sig2 = pd.DataFrame({"decline_rate": ypos.groupby(low_ctr).mean().round(3),
                     "n": ypos.groupby(low_ctr).size()})
sig2.index = ["ctr ok for band", "ctr < 50% of band median"]
print("Signal 2 — CTR vs position band (n with position data:", len(pos), "):")
print(sig2.to_string())
print("\nVerdict: CONFIRMED (directional): +6.6 pp decline for under-clicking pages")

Signal 2 — CTR vs position band (n with position data: 28795 ):
                          decline_rate      n
ctr ok for band                  0.547  21159
ctr < 50% of band median         0.613   7636

Verdict: CONFIRMED (directional): +6.6 pp decline for under-clicking pages


## 2. Build the ranked queue (writes the CSV)

The rule as transparent code — flags multiplied, no fitted weights, ranked by impressions.
The queue CSV goes to `work/outputs/baseline_action_score.csv` (out of git by design; the
CI leak-guard blocks data files and the notebook regenerates it every run). The run's
receipts — the metrics — go to `work/outputs/baseline_metrics.json`, which IS committed.

Honest depth note: only **35 pages** score above zero. The rule's cell is precise
(**0.743** decline inside it vs 0.542 base) but it *starves* — it cannot fill a 50-page
queue. That starvation is exactly what the Week-5 model must fix, and the baseline stays
frozen from here.

In [3]:
# The rule, the queue, the CSV, the receipts.
RULE = "days_since_last_update >= 180 AND impressions_90d >= 100, ranked by impressions_90d"
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 100).astype(int)
df["score"] = stale * visible * df["impressions_90d"]        # readable on purpose
df["reason_code"] = np.where(df["score"] > 0, "STALE_VISIBLE", "")
df["action"] = np.where(df["score"] > 0, "REFRESH", "")

queue = (df.loc[:, ["content_id", "client_id", "content_type", "days_since_last_update",
                    "impressions_90d", "avg_position", "ctr", "score", "reason_code", "action"]]
           .sort_values("score", ascending=False).reset_index(drop=True))
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

n_scored = int((df["score"] > 0).sum())
rng = np.random.default_rng(42)
metrics = {
    "rule": RULE,
    "n_scored": n_scored,
    "cell_decline_rate": round(float(y[df["score"] > 0].mean()), 3),
    "precision_at_10": round(precision_at_k(df["score"], y, 10), 3),
    "precision_at_20": round(precision_at_k(df["score"], y, 20), 3),
    "precision_at_35_full_depth": round(precision_at_k(df["score"], y, n_scored), 3),
    "precision_at_50_padded": round(precision_at_k(df["score"], y, 50), 3),
    "random_floor_p50_200_shuffles": round(float(np.mean(
        [precision_at_k(rng.random(len(df)), y, 50) for _ in range(200)])), 3),
    "base_rate_dummy_floor": round(float(y.mean()), 3),
}
import json as _json
with open("work/outputs/baseline_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"queue written: work/outputs/baseline_action_score.csv ({len(queue):,} rows, {n_scored} scored)")
print(_json.dumps(metrics, indent=2))
print("\nCaveat printed with the number: ranks past", n_scored,
      "are unscored ties — precision_at_50_padded's tail is arbitrary by construction.")

queue written: work/outputs/baseline_action_score.csv (30,000 rows, 35 scored)
{
  "rule": "days_since_last_update >= 180 AND impressions_90d >= 100, ranked by impressions_90d",
  "n_scored": 35,
  "cell_decline_rate": 0.743,
  "precision_at_10": 1.0,
  "precision_at_20": 0.9,
  "precision_at_35_full_depth": 0.743,
  "precision_at_50_padded": 0.64,
  "random_floor_p50_200_shuffles": 0.534,
  "base_rate_dummy_floor": 0.542
}

Caveat printed with the number: ranks past 35 are unscored ties — precision_at_50_padded's tail is arbitrary by construction.


## 3. Top-20 review

One line per pick: the action, why it's there, a confidence note, and what would make it
wrong. The `declining` column is the evaluation label shown for transparency — the rule
never saw it. Reading done with a skeptic's eye; the weak rows get named in §4.

In [4]:
# Top 20 with per-row review lines built from the row's own numbers.
top = queue.head(20).copy()
top["declining"] = y.loc[df.sort_values("score", ascending=False).index[:20]].values

def review(r):
    why = f"untouched {int(r.days_since_last_update)}d yet {int(r.impressions_90d):,} impressions/90d"
    conf = "high" if r.impressions_90d >= 4000 else ("medium" if r.impressions_90d >= 800 else "low")
    if r.ctr >= 0.5:
        wrong = "CTR is healthy for its reach - page may be fine; refresh could be wasted effort"
    elif r.avg_position <= 10:
        wrong = "already ranks page-1; a clumsy rewrite risks the ranking it still holds"
    else:
        wrong = "if this is an evergreen/reference page, stale-by-date is not stale-by-content"
    return pd.Series({"why": why, "confidence": conf, "what_would_make_it_wrong": wrong})

top = pd.concat([top[["content_id", "action", "declining"]],
                 top.apply(review, axis=1)], axis=1)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
print(top.to_string(index=False))
print(f"\n{int(top['declining'].sum())}/20 of the reviewed picks are in fact declining.")

          content_id  action  declining                                       why confidence                                                        what_would_make_it_wrong
content_cf56e2e2e282 REFRESH          1 untouched 194d yet 61,678 impressions/90d       high   if this is an evergreen/reference page, stale-by-date is not stale-by-content
content_7368877ea310 REFRESH          1 untouched 194d yet 59,472 impressions/90d       high   if this is an evergreen/reference page, stale-by-date is not stale-by-content
content_1bfaa38ff26c REFRESH          1 untouched 194d yet 25,715 impressions/90d       high   if this is an evergreen/reference page, stale-by-date is not stale-by-content
content_0a91db491d14 REFRESH          1 untouched 193d yet 13,299 impressions/90d       high   if this is an evergreen/reference page, stale-by-date is not stale-by-content
content_5feee3994adb REFRESH          1  untouched 194d yet 7,812 impressions/90d       high   if this is an evergreen/reference page, 

## 4. Weak picks + leakage check

**Weak picks found (looking harder, as required):**

- **Rank 12** (`content_bdbec75c1148`) and **rank 19** (`content_b65fe2792b44`) are **not
  declining** — rank 19's CTR (0.54, ×100 scale) is healthy for its band; "stale by date"
  is doing all the work and the page disagrees.
- **Rank 20** (`content_ba00ffc6318c`): CTR 4.93 is ~30× the panel median — a small page
  that already over-performs; a refresh is likely wasted editor time even though it's
  labeled declining this window.
- **Monoculture:** all 20 picks are the same content type (keyword articles), because
  ranking by raw impressions favors that type's volume. The queue never surfaces other
  types — a coverage bias the model needs to beat.
- **Threshold fragility:** the 180-day cut sits in a staleness band whose *marginal* table
  (§1) is non-monotone; the rule survives only through the visibility interaction — and it
  starves after 35 pages.

**Leakage check:** the rule reads exactly two inputs — `days_since_last_update` and
`impressions_90d` — both trailing-window facts. Assertions below prove no label-derived
column (`trend_direction`, `trend_pct`, `is_declining_label`) and no future window touched
the score. The starter release carries no FlyRank product flags at all, by design.

In [5]:
# Leakage guard: the rule's inputs, checked against the forbidden set.
RULE_INPUTS = {"days_since_last_update", "impressions_90d"}
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label"}

assert not RULE_INPUTS & FORBIDDEN, "leak: rule reads a label-derived column"
assert RULE_INPUTS <= set(df.columns), "rule input missing from the frame"
product_flag_cols = [c for c in df.columns if "flag" in c.lower() or "optimiz" in c.lower()]
print("rule inputs:", sorted(RULE_INPUTS))
print("forbidden set untouched:", sorted(FORBIDDEN))
print("product-flag columns present in starter data:", product_flag_cols or "none (by design)")
print("\nBaseline frozen: P@10 1.0 / P@20 0.90 at the head, 0.743 at full 35-row depth,")
print("starving past that - this is the number the Week-5 model must beat at real depth.")

rule inputs: ['days_since_last_update', 'impressions_90d']
forbidden set untouched: ['is_declining_label', 'trend_direction', 'trend_pct']
product-flag columns present in starter data: none (by design)

Baseline frozen: P@10 1.0 / P@20 0.90 at the head, 0.743 at full 35-row depth,
starving past that - this is the number the Week-5 model must beat at real depth.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.